# Forecast Resilience of ARIMA Models Under Macroeconomic Shocks

**Mentor-approved undergraduate research** testing whether standard ARIMA models lose forecast accuracy during macroeconomic volatility, and whether incorporating macroeconomic indicators improves resilience.

**Data:** 4 FRED time series (retail sales, CPI/inflation, federal funds rate, USD index), Jan 2019 - Dec 2024, split into stable (2019-20) / volatile (2021-22) / recovery (2023-24) periods.

## Key Findings
1. **Shock type mattered more than shock category** - the sudden COVID shock (sitting inside the "stable" 2019-20 period) degraded forecast accuracy far more than the gradual 2021-22 inflationary volatility the framework was built around.
2. **Macro-awareness helps, but only when applied selectively** - one well-chosen macroeconomic variable improved forecast resilience; adding all three at once did not.
3. **Model complexity has real limits given data availability** - a fully-specified VAR became infeasible on this dataset, supporting simpler, selective models when historical data is limited.


## 1. Data Loading & Cleaning

Cleaning and merging of all 4 datasets.
Goal: one clean table, monthly, Jan 2019-Dec 2024, with:
RSAFS (Retail sales dataset) : 'retail_sales' (feature to be forecasted)
CPIAUCSL (Inflation Data): 'inflation_yoy' (added feature for crosschecking prediction accuracies)
FEDFUNDS (interest rates) : 'interest_rate'
TWEXBGSMTH (Dollar index ) :'dollar_index'


In [ ]:
import pandas as pd 

# STEP 1: Load each file
Each FRED file has 2 columns: 'observation_date' and the value.
We tell pandas to treat 'observation_date' as an actual date,
not just text, so we can filter and merge by date later.


In [ ]:
rsafs = pd.read_csv(r"data/RSAFS.csv", parse_dates=["observation_date"],dayfirst=True)

cpi = pd.read_csv(r"data/CPIAUCSL.csv", parse_dates=["observation_date"],dayfirst=True) 

fedfunds = pd.read_csv(r"data/FEDFUNDS.csv", parse_dates=["observation_date"],dayfirst=True)

dollar = pd.read_csv(r"data/TWEXBGSMTH.csv", parse_dates=["observation_date"])

STEP 2: Rename columns to clear, human-readable names

In [ ]:
rsafs = rsafs.rename(columns={"observation_date": "date", "RSAFS": "retail_sales"})

cpi = cpi.rename(columns={"observation_date": "date", "CPIAUCSL": "cpi_index"})

fedfunds = fedfunds.rename(columns={"observation_date": "date", "FEDFUNDS": "interest_rate"})

dollar = dollar.rename(columns={"observation_date": "date", "TWEXBGSMTH": "dollar_index"})


#STEP 3: Turn raw CPI into "inflation" (year-over-year % change)
CPI on its own (e.g. "252.5") means nothing to a normal reader.
What matters is: how much did prices rise compared to 12 months ago?
pct_change(12) does exactly that on monthly data (12 months back).

In [ ]:
cpi = cpi.sort_values("date")
cpi["inflation_yoy"] = cpi["cpi_index"].pct_change(12) * 100
cpi = cpi[["date", "inflation_yoy"]]   #since we only need the transformed column now 

Quick sanity checks on each source file before merging — shape, date range, duplicate dates, and missing values.

In [ ]:
print(rsafs.shape)
print(rsafs.head())
print(rsafs.tail())
print(rsafs['date'].dt.day.unique())
print(rsafs['date'].duplicated().sum())
print(rsafs.isnull().sum())

In [ ]:
print(cpi.shape)
print(cpi.head())
print(cpi.tail())
print(cpi['date'].dt.day.unique())
print(cpi['date'].duplicated().sum())
print(cpi.isnull().sum())

In [ ]:
print(fedfunds.shape)
print(fedfunds.head())
print(fedfunds.tail())
print(fedfunds['date'].dt.day.unique())
print(fedfunds['date'].duplicated().sum())
print(fedfunds.isnull().sum())

In [ ]:
print(dollar.shape)
print(dollar.head())
print(dollar.tail())
print(dollar['date'].dt.day.unique())
print(dollar['date'].duplicated().sum())
print(dollar.isnull().sum())

STEP 4: Merge everything into a single table, matched by date
"outer" merge keeps every date from every file, even if one file
is missing that date (like FEDFUNDS after Jan 2024) - this lets us
SEE the gap clearly instead of silently losing data.

In [ ]:
merged = rsafs.merge(cpi, on="date", how="outer")
merged = merged.merge(fedfunds, on="date", how="outer")
merged = merged.merge(dollar, on="date", how="outer")
merged = merged.sort_values("date").reset_index(drop=True)

STEP 5: Keep only our research window: Jan 2019 - Dec 2024
(CPI file has extra years beyond 2024 that we don't need)

In [ ]:
merged = merged[(merged["date"] >= "2019-01-01") & (merged["date"] <= "2024-12-31")]
merged = merged.reset_index(drop=True)

In [ ]:
print(merged['date'].min())
print(merged['date'].max())

In [ ]:
print(merged.shape)

print("RSAFS:", rsafs.shape)
print("CPI:", cpi.shape)
print("FEDFUNDS:", fedfunds.shape)
print("Dollar:", dollar.shape)

STEP 6: Tag each row with its research sub-period

In [ ]:
def tag_period(date):
    if date <= pd.Timestamp("2020-12-31"):
        return "stable"
    elif date <= pd.Timestamp("2022-12-31"):
        return "volatile"
    else:
        return "recovery"

merged["period"] = merged["date"].apply(tag_period)


STEP 7: Report data quality - show any missing values clearly

In [ ]:
print("=== Shape of final merged table ===")
print(merged.shape, "\n")

print("=== Missing values per column ===")
print(merged.isna().sum(), "\n")

print("=== Rows with missing interest_rate (the FEDFUNDS gap) ===")
print(merged[merged["interest_rate"].isna()][["date", "interest_rate"]], "\n")

print("=== Row count per period ===")
print(merged["period"].value_counts(), "\n")

print("=== First 5 rows ===")
print(merged.head(), "\n")

print("=== Last 5 rows ===")
print(merged.tail(), "\n")

**Data-quality investigation:** the merge above showed missing `interest_rate` values in recent months. Before assuming this was a data problem, we went back to inspect the raw source files directly rather than trusting a first-pass `isnull()` check.

In [ ]:
import pandas as pd

# Load each file individually and inspect BEFORE merging
fedfunds_check = pd.read_csv(r"data/FEDFUNDS.csv", parse_dates=["observation_date"])
print("FEDFUNDS shape:", fedfunds_check.shape)
print(fedfunds_check.head(10))
print(fedfunds_check.tail(5))

cpi_check = pd.read_csv(r"data/CPIAUCSL.csv", parse_dates=["observation_date"])
print("\nCPI shape:", cpi_check.shape)
print(cpi_check.head(10))

In [ ]:
print(merged.shape)
print(merged.isna().sum())
print(merged['period'].value_counts())

In [ ]:
print("RSAFS:", rsafs.shape, rsafs['date'].dt.day.unique(), rsafs['date'].duplicated().sum())
print("CPI:", cpi.shape, cpi['date'].dt.day.unique(), cpi['date'].duplicated().sum())
print("FEDFUNDS:", fedfunds.shape, fedfunds['date'].dt.day.unique(), fedfunds['date'].duplicated().sum())
print("Dollar:", dollar.shape, dollar['date'].dt.day.unique(), dollar['date'].duplicated().sum())
print()
print("MERGED:", merged.shape)
print(merged.isna().sum())

In [ ]:
with open(r"data/TWEXBGSMTH.csv", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 15:
            break

In [ ]:
print(merged.shape)
print(merged.isna().sum())
print(merged['period'].value_counts())

## 2. Stationarity Testing (ADF)

We run a statistical test called the Augmented Dickey-Fuller (ADF) test on your retail_sales numbers. In plain terms: this test asks "does this data have a clear trend, or does it bounce around a stable average?" If it has a trend, we need to difference it (d=1) before ARIMA can work well.

ADF (Augmented Dickey-Fuller) Test
-----------------------------------
Purpose: Check whether our retail_sales data has a trend, so we know
what 'd' value to use in our ARIMA(p, d, q) model.
 
Rule for reading the result:
- p-value < 0.05  -> data has NO significant trend (stationary) -> d = 0
- p-value >= 0.05 -> data HAS a trend (non-stationary)          -> d = 1

In [ ]:
from statsmodels.tsa.stattools import adfuller

df = merged

print("=== ADF Test for all three periods ===")
for period_name in ["stable", "volatile", "recovery"]:
    period_data = df[df["period"] == period_name].sort_values("date")["retail_sales"]
    result = adfuller(period_data)
    verdict = "d = 0 (stationary)" if result[1] < 0.05 else "d = 1 (non-stationary, has trend)"
    print(f"{period_name:10s} -> ADF stat = {result[0]:.4f}, p-value = {result[1]:.4f} -> {verdict}")

## 3. Baseline ARIMA Modeling

In [ ]:
!pip install pmdarima
!pip install --upgrade statsmodels pmdarima

In [ ]:
import pmdarima as pm

print("=== Auto ARIMA results for all three periods ===\n")

for period_name in ["stable", "volatile", "recovery"]:
    period_data = df[df["period"] == period_name].sort_values("date")["retail_sales"]
    
    model = pm.auto_arima(
        period_data,
        d=1,                  # we already know d=1 from the ADF test
        seasonal=False,       # no seasonal component for now (24 months is short for seasonality)
        trace=True,           # prints out every combination it tries
        suppress_warnings=True,
        stepwise=True         # smart search instead of testing every possible combo
    )
    
    print(f"\n{period_name.upper()} — Best model: {model.order}")
    print(model.summary())
    print("\n" + "="*80 + "\n")

| Period   | Best model (p,d,q) | AIC (lower = better fit) |
|----------|---------------------|--------------------------|
| Stable   | ARIMA(0,1,0)        | 533.6 |
| Volatile | ARIMA(0,1,2)        | 476.2 |
| Recovery | ARIMA(0,1,2)        | 445.0 |


## 4. Walk-Forward Validation

In [ ]:
import numpy as np
from statsmodels.tsa.arima.model import ARIMA

# Best (p,d,q) orders found using auto_arima for each period
best_orders = {
    "stable": (0, 1, 0),
    "volatile": (0, 1, 2),
    "recovery": (0, 1, 2),
}

INITIAL_TRAIN_SIZE = 12  # months of history before we start testing

results_summary = []
walk_forward_data = {}  # store per-period actuals/predictions/dates for plotting

for period_name in ["stable", "volatile", "recovery"]:
    print(f"\n{'='*60}")
    print(f"PERIOD: {period_name.upper()}")
    print(f"{'='*60}")

    period_dates = df[df["period"] == period_name].sort_values("date")["date"].reset_index(drop=True)
    period_data = df[df["period"] == period_name].sort_values("date")["retail_sales"].reset_index(drop=True)
    order = best_orders[period_name]
    actuals, predictions = [], []

    for i in range(INITIAL_TRAIN_SIZE, len(period_data)):
        train = period_data.iloc[:i]
        actual_value = period_data.iloc[i]

        model = ARIMA(train, order=order)
        fitted_model = model.fit()
        forecast = fitted_model.forecast(steps=1)
        predicted_value = forecast.iloc[0]

        actuals.append(actual_value)
        predictions.append(predicted_value)
        print(f"Month {i+1:2d}: actual = {actual_value:>10,.0f}   predicted = {predicted_value:>10,.0f}")

    actuals = np.array(actuals)
    predictions = np.array(predictions)
    rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
    mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

    print(f"\n{period_name.upper()} RESULTS: RMSE = {rmse:,.2f}   MAPE = {mape:.2f}%")
    results_summary.append({"period": period_name, "order": order, "rmse": rmse, "mape": mape, "n_predictions": len(actuals)})
    walk_forward_data[period_name] = {
        "dates": period_dates.iloc[INITIAL_TRAIN_SIZE:].reset_index(drop=True),
        "actuals": actuals,
        "predictions": predictions,
    }

print(f"\n{'='*60}\nSUMMARY TABLE\n{'='*60}")
summary_df = pd.DataFrame(results_summary)
print(summary_df.to_string(index=False))

### Chart: Actual vs. Predicted Retail Sales, by Period

The stable period's forecast line should visibly diverge from actuals during the COVID-crash months (shaded), while the rest of the stable period tracks closely — this is the chart version of the MAPE finding above.

In [ ]:
import matplotlib.pyplot as plt

covid_start, covid_end = pd.Timestamp("2020-03-01"), pd.Timestamp("2020-06-01")

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)

for ax, period_name in zip(axes, ["stable", "volatile", "recovery"]):
    d = walk_forward_data[period_name]
    ax.plot(d["dates"], d["actuals"], label="Actual", color="black", linewidth=1.5)
    ax.plot(d["dates"], d["predictions"], label="Predicted", color="tab:red",
            linestyle="--", linewidth=1.5)

    if period_name == "stable":
        ax.axvspan(covid_start, covid_end, color="orange", alpha=0.2,
                   label="COVID-crash months")

    ax.set_title(f"{period_name.capitalize()} period")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=8)

fig.suptitle("Actual vs. Predicted Retail Sales (Walk-Forward Validation)")
fig.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Shock-Type Analysis: COVID Crash vs. Labeled Volatility

The "stable" period (2019-2020) technically includes the COVID crash months (March-June 2020). Re-running walk-forward validation while explicitly excluding those months isolates how much of the "stable" period's error actually came from the COVID shock itself, versus ordinary noise.

In [ ]:
covid_crash_dates = pd.to_datetime(["2020-03-01", "2020-04-01", "2020-05-01", "2020-06-01"])

stable_data = df[df["period"] == "stable"].sort_values("date").reset_index(drop=True)
order = best_orders["stable"]

actuals_excl, predictions_excl = [], []

for i in range(INITIAL_TRAIN_SIZE, len(stable_data)):
    train = stable_data["retail_sales"].iloc[:i]
    actual_value = stable_data["retail_sales"].iloc[i]
    actual_date = stable_data["date"].iloc[i]

    model = ARIMA(train, order=order)
    fitted_model = model.fit()
    forecast = fitted_model.forecast(steps=1)
    predicted_value = forecast.iloc[0]

    if actual_date in covid_crash_dates:
        print(f"{actual_date.date()}: EXCLUDED (COVID crash month)")
        continue

    actuals_excl.append(actual_value)
    predictions_excl.append(predicted_value)

actuals_excl = np.array(actuals_excl)
predictions_excl = np.array(predictions_excl)
rmse_excl = np.sqrt(np.mean((actuals_excl - predictions_excl) ** 2))
mape_excl = np.mean(np.abs((actuals_excl - predictions_excl) / actuals_excl)) * 100

print(f"\nStable period EXCLUDING COVID months: RMSE = {rmse_excl:,.2f}   MAPE = {mape_excl:.2f}%")
print(f"Stable period INCLUDING COVID months: RMSE = 34,824.10   MAPE = 4.78%")

## 6. ARIMAX Extension: Incorporating Macroeconomic Indicators

Does adding macro context (inflation, interest rates, the dollar index) improve forecast resilience? Tested three ways: all three variables together, one variable at a time, and inflation alone.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA

exog_cols = ["inflation_yoy", "interest_rate", "dollar_index"]

# Best (p,d,q) orders found via auto_arima WITH exogenous variables
best_orders_arimax = {
    "stable": (0, 1, 0),
    "volatile": (1, 1, 3),
    "recovery": (2, 1, 0),
}

# Baseline results, already computed, for direct comparison
baseline_results = {
    "stable": {"rmse": 34824.10, "mape": 4.78},
    "volatile": {"rmse": 7449.57, "mape": 1.02},
    "recovery": {"rmse": 4495.04, "mape": 0.56},
}

INITIAL_TRAIN_SIZE = 12

arimax_results = []

for period_name in ["stable", "volatile", "recovery"]:
    print(f"\n{'='*60}")
    print(f"PERIOD: {period_name.upper()} (ARIMAX)")
    print(f"{'='*60}")

    period_df = df[df["period"] == period_name].sort_values("date").reset_index(drop=True)
    y = period_df["retail_sales"]
    X = period_df[exog_cols]

    order = best_orders_arimax[period_name]
    actuals, predictions = [], []

    for i in range(INITIAL_TRAIN_SIZE, len(period_df)):
        y_train = y.iloc[:i]
        X_train = X.iloc[:i]
        X_next = X.iloc[i:i+1]
        actual_value = y.iloc[i]

        model = ARIMA(y_train, order=order, exog=X_train)
        fitted_model = model.fit()

        forecast = fitted_model.forecast(steps=1, exog=X_next)
        predicted_value = forecast.iloc[0]

        actuals.append(actual_value)
        predictions.append(predicted_value)
        print(f"Month {i+1:2d}: actual = {actual_value:>10,.0f}   predicted = {predicted_value:>10,.0f}")

    actuals = np.array(actuals)
    predictions = np.array(predictions)
    rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
    mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

    print(f"\n{period_name.upper()} ARIMAX RESULTS: RMSE = {rmse:,.2f}   MAPE = {mape:.2f}%")

    arimax_results.append({
        "period": period_name, "order": order, "rmse": rmse, "mape": mape, "n_predictions": len(actuals)
    })

print(f"\n{'='*70}")
print("COMPARISON: BASELINE (ARIMA) vs ENHANCED (ARIMAX)")
print(f"{'='*70}")

comparison_rows = []
for r in arimax_results:
    p = r["period"]
    base_mape = baseline_results[p]["mape"]
    enh_mape = r["mape"]
    improvement = base_mape - enh_mape
    comparison_rows.append({
        "period": p, "baseline_mape": base_mape, "arimax_mape": round(enh_mape, 2),
        "improvement_pct_points": round(improvement, 2),
        "did_it_help": "YES" if improvement > 0 else "NO"
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))
comparison_df.to_csv("baseline_vs_arimax_comparison.csv", index=False)

In [ ]:
print(f"\n{'='*70}")
print("SINGLE-VARIABLE TEST: VOLATILE PERIOD")
print(f"{'='*70}")

single_var_results = []

volatile_df = df[df["period"] == "volatile"].sort_values("date").reset_index(drop=True)
y = volatile_df["retail_sales"]

for var in ["inflation_yoy", "interest_rate", "dollar_index"]:
    print(f"\n--- Testing with only: {var} ---")

    X = volatile_df[[var]]

    # Re-find best order for this single-variable version
    order = pm.auto_arima(y, X=X, d=1, seasonal=False, trace=False, suppress_warnings=True, stepwise=True).order
    print(f"Best order for {var}: {order}")

    actuals, predictions = [], []

    for i in range(INITIAL_TRAIN_SIZE, len(volatile_df)):
        y_train = y.iloc[:i]
        X_train = X.iloc[:i]
        X_next = X.iloc[i:i+1]
        actual_value = y.iloc[i]

        model = ARIMA(y_train, order=order, exog=X_train)
        fitted_model = model.fit()
        forecast = fitted_model.forecast(steps=1, exog=X_next)
        predicted_value = forecast.iloc[0]

        actuals.append(actual_value)
        predictions.append(predicted_value)

    actuals = np.array(actuals)
    predictions = np.array(predictions)
    rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
    mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

    print(f"{var} ALONE: RMSE = {rmse:,.2f}   MAPE = {mape:.2f}%")
    single_var_results.append({"variable": var, "order": order, "rmse": rmse, "mape": mape})

print(f"\n{'='*70}")
print("SUMMARY: SINGLE-VARIABLE COMPARISON (VOLATILE PERIOD)")
print(f"{'='*70}")
single_var_df = pd.DataFrame(single_var_results)
print(single_var_df.to_string(index=False))
print(f"\nFor reference:")
print(f"  Baseline (no macro vars):        MAPE = 1.02%")
print(f"  All 3 macro vars together:       MAPE = 2.04%")

single_var_df.to_csv("volatile_single_variable_test.csv", index=False)

In [ ]:
print(f"\n{'='*70}")
print("INFLATION-ONLY MODEL: ALL THREE PERIODS")
print(f"{'='*70}")

inflation_results = []

for period_name in ["stable", "volatile", "recovery"]:
    print(f"\n--- {period_name.upper()} (inflation_yoy only) ---")

    period_df = df[df["period"] == period_name].sort_values("date").reset_index(drop=True)
    y = period_df["retail_sales"]
    X = period_df[["inflation_yoy"]]

    # Find best order for this period using only inflation as the exogenous variable
    order = pm.auto_arima(y, X=X, d=1, seasonal=False, trace=False, suppress_warnings=True, stepwise=True).order
    print(f"Best order: {order}")

    actuals, predictions = [], []

    for i in range(INITIAL_TRAIN_SIZE, len(period_df)):
        y_train = y.iloc[:i]
        X_train = X.iloc[:i]
        X_next = X.iloc[i:i+1]
        actual_value = y.iloc[i]

        model = ARIMA(y_train, order=order, exog=X_train)
        fitted_model = model.fit()
        forecast = fitted_model.forecast(steps=1, exog=X_next)
        predicted_value = forecast.iloc[0]

        actuals.append(actual_value)
        predictions.append(predicted_value)

    actuals = np.array(actuals)
    predictions = np.array(predictions)
    rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
    mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

    print(f"{period_name.upper()} (inflation only): RMSE = {rmse:,.2f}   MAPE = {mape:.2f}%")
    inflation_results.append({"period": period_name, "order": order, "rmse": rmse, "mape": mape})

# -----------------------------------------------------------------
# Final combined comparison: Baseline vs All-3-macro-vars vs Inflation-only
# -----------------------------------------------------------------
print(f"\n{'='*70}")
print("FINAL COMPARISON: BASELINE vs ARIMAX (all 3 vars) vs INFLATION-ONLY")
print(f"{'='*70}")

# Fill in your actual baseline and all-3-var MAPE values from earlier results
baseline_mape = {"stable": 4.78, "volatile": 1.02, "recovery": 0.56}
all3_mape = {"stable": 3.89, "volatile": 2.31, "recovery": 0.54}

final_comparison = []
for r in inflation_results:
    p = r["period"]
    final_comparison.append({
        "period": p,
        "baseline_mape": baseline_mape[p],
        "all_3_vars_mape": all3_mape[p],
        "inflation_only_mape": round(r["mape"], 2),
    })

final_df = pd.DataFrame(final_comparison)
print(final_df.to_string(index=False))

final_df.to_csv("final_model_comparison.csv", index=False)
print("\nSaved final_model_comparison.csv")

### Chart: MAPE Comparison — Baseline vs. All-3-Variable ARIMAX vs. Inflation-Only

Visualizes the selective-variable finding: inflation-only sits at or below both other bars in every period.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

periods = ["stable", "volatile", "recovery"]
baseline_vals = [baseline_mape[p] for p in periods]
all3_vals = [all3_mape[p] for p in periods]
inflation_vals = [round(r["mape"], 2) for r in inflation_results]

x = np.arange(len(periods))
width = 0.25

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x - width, baseline_vals, width, label="Baseline (no macro vars)")
ax.bar(x, all3_vals, width, label="All 3 macro vars")
ax.bar(x + width, inflation_vals, width, label="Inflation only")

ax.set_xticks(x)
ax.set_xticklabels([p.capitalize() for p in periods])
ax.set_ylabel("MAPE (%)")
ax.set_title("Forecast Error by Model Specification")
ax.legend()
fig.tight_layout()
plt.savefig("mape_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
print(f"\n{'='*70}")
print("SINGLE-VARIABLE TEST: ALL VARIABLES, ALL PERIODS")
print(f"{'='*70}")

all_single_var_results = []

for period_name in ["stable", "volatile", "recovery"]:
    period_df = df[df["period"] == period_name].sort_values("date").reset_index(drop=True)
    y = period_df["retail_sales"]

    for var in ["inflation_yoy", "interest_rate", "dollar_index"]:
        print(f"\n--- {period_name.upper()} — {var} only ---")

        X = period_df[[var]]
        order = pm.auto_arima(y, X=X, d=1, seasonal=False, trace=False, suppress_warnings=True, stepwise=True).order

        actuals, predictions = [], []

        for i in range(INITIAL_TRAIN_SIZE, len(period_df)):
            y_train = y.iloc[:i]
            X_train = X.iloc[:i]
            X_next = X.iloc[i:i+1]
            actual_value = y.iloc[i]

            model = ARIMA(y_train, order=order, exog=X_train)
            fitted_model = model.fit()
            forecast = fitted_model.forecast(steps=1, exog=X_next)
            predicted_value = forecast.iloc[0]

            actuals.append(actual_value)
            predictions.append(predicted_value)

        actuals = np.array(actuals)
        predictions = np.array(predictions)
        rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
        mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

        print(f"{period_name} — {var}: RMSE = {rmse:,.2f}   MAPE = {mape:.2f}%")
        all_single_var_results.append({
            "period": period_name, "variable": var, "order": order, "rmse": rmse, "mape": round(mape, 2)
        })

print(f"\n{'='*70}")
print("FULL SINGLE-VARIABLE COMPARISON TABLE")
print(f"{'='*70}")
full_results_df = pd.DataFrame(all_single_var_results)
pivot_table = full_results_df.pivot(index="period", columns="variable", values="mape")
print(pivot_table.to_string())

full_results_df.to_csv("all_single_variable_results.csv", index=False)
pivot_table.to_csv("single_variable_mape_pivot.csv")
print("\nSaved all_single_variable_results.csv and single_variable_mape_pivot.csv")

## 7. VAR Model: Full Multivariate System

As a final test of model complexity, we fit a full Vector Autoregression (VAR) treating all four variables as jointly endogenous — first confirming stationarity of each.

In [ ]:
from statsmodels.tsa.stattools import adfuller

print(f"\n{'='*70}")
print("ADF TEST: ALL 4 VARIABLES, ALL PERIODS")
print(f"{'='*70}")

variables = ["retail_sales", "inflation_yoy", "interest_rate", "dollar_index"]

for period_name in ["stable", "volatile", "recovery"]:
    print(f"\n--- {period_name.upper()} ---")
    period_df = df[df["period"] == period_name].sort_values("date")
    for var in variables:
        result = adfuller(period_df[var])
        verdict = "STATIONARY" if result[1] < 0.05 else "NON-STATIONARY (needs differencing)"
        print(f"  {var:15s} -> p-value = {result[1]:.4f} -> {verdict}")

In [ ]:
from statsmodels.tsa.api import VAR

print(f"\n{'='*70}")
print("VAR MODEL: ALL THREE PERIODS")
print(f"{'='*70}")

var_cols = ["retail_sales", "inflation_yoy", "interest_rate", "dollar_index"]

for period_name in ["stable", "volatile", "recovery"]:
    print(f"\n--- {period_name.upper()} ---")

    period_df = df[df["period"] == period_name].sort_values("date")[var_cols].reset_index(drop=True)

    # Difference all variables once, uniformly, for consistency
    period_diff = period_df.diff().dropna()

    # Fit VAR and let it select the best lag order using AIC
    model = VAR(period_diff)

    try:
        lag_selection = model.select_order(maxlags=4)
        best_lag = lag_selection.aic
        print(f"Best lag order (by AIC): {best_lag}")
    except ValueError as e:
        # This is a finding, not a bug: with only ~24 monthly observations and
        # 4 variables, a fully-specified VAR needs more parameters than the
        # data can support. This directly evidences the paper's third
        # conclusion -- model complexity has real limits given data
        # availability. We fall back to a minimal lag order to still produce
        # a comparison point, rather than letting the notebook crash.
        print(f"VAR order selection infeasible at maxlags=4: {e}")
        print("This confirms the paper's third finding: a fully-specified VAR "
              "is infeasible on this dataset given how little history is "
              "available per period. Falling back to lag order 1.")
        best_lag = 1

    fitted = model.fit(best_lag if best_lag > 0 else 1)
    print(fitted.summary())


## Conclusions

1. **Shock type mattered more than shock category** — the sudden COVID shock hurt forecast accuracy far more than the gradual inflationary volatility this study was originally designed to test.
2. **Macro-awareness helps, but only when applied selectively** — one well-chosen variable improved resilience; naively including every macro variable did not.
3. **Model complexity has real limits given data availability** — VAR's infeasibility on this dataset itself makes the case for simpler, practical models when historical data is limited.